# Cut Before — Problem 1: Find the Promos

**Atit Kumar Satsangi | 230241 | Civil Engineering**

Upload a new timestamped screenplay, connect a model API and run the cells in order. No example story or saved model answer is preloaded.

**Output:** ranked 20–60 second promo suggestions, each with episode, start/end timestamps, hook in the first 3 seconds, unresolved cliffhanger and a reason. This version selects one continuous segment per retained episode. It only finds moments: **no video editing, music or text overlays**.

## 1. Set up
Run this cell once. It downloads the tested Python code; it does not load a story or call AI. The source code is available in the [repository](https://github.com/atits23/cut-before-kuku-assignment).

In [ ]:
#@title 1. Set up the promo finder
from pathlib import Path
import hashlib
import urllib.request

SOURCE = "https://raw.githubusercontent.com/atits23/cut-before-kuku-assignment/9363b9921fd9dd47940c553c9b849e16051a651f/"
EXPECTED = {'cut_before.py': '1aa23bdcf8ec3c14909423c24dc8e59dfb31f92fa904866f376ba21ccda13729', 'api_review.py': '5645a00b12cbff6e8520f9957a8d54b572eb3f6b5cd9692dba7e86499f0cdda5'}
for name, expected in EXPECTED.items():
    with urllib.request.urlopen(SOURCE + name, timeout=30) as response:
        source = response.read()
    if hashlib.sha256(source).hexdigest() != expected:
        raise ValueError("Setup file did not match the tested version. Stop and reopen the notebook.")
    Path(name).write_bytes(source)
import importlib
import cut_before
import api_review
importlib.reload(cut_before)
importlib.reload(api_review)
print("Ready. Upload your screenplay below. No story has been loaded.")


## 2. Upload a story
Use one `.txt` screenplay or a `.zip` containing episode `.txt` files. Use the assignment’s timestamped format: an episode-number header and event blocks with `Start: (MM:SS.cc) | End: (MM:SS.cc)`. PDFs and video are not supported. Upload the full pack to review every episode.

In [ ]:
#@title 2. Upload your screenplay TXT or ZIP
from google.colab import files
import uuid

selected_input = None
episode_count = 0
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one screenplay TXT or one ZIP containing all episode TXT files.")
filename, contents = next(iter(uploaded.items()))
suffix = Path(filename).suffix.lower()
if suffix not in (".txt", ".zip"):
    raise ValueError("Use a timestamped TXT screenplay or ZIP of episode TXT files.")
input_folder = Path("inputs") / uuid.uuid4().hex[:12]
input_folder.mkdir(parents=True)
candidate = input_folder / ("screenplay" + suffix)
candidate.write_bytes(contents)
episodes, input_hash = cut_before.load_pack(candidate)
selected_input = candidate
episode_count = len(episodes)
print(f"Loaded {filename}: {episode_count} episodes, {sum(len(e['beats']) for e in episodes)} timed events.")
print(f"The next step will review every episode: {episode_count} API requests.")


## 3. API settings
Use the **API base URL, model ID and key supplied to you**. Choose `responses` for OpenAI Responses or `chat_completions` for a compatible API using Bearer authentication. The base URL excludes `/responses` and `/chat/completions`. The key is entered privately in step 4. Changing the model does not change the parser or ranking.

Only send scripts to a provider approved for them. The run makes one request per uploaded episode; provider charges may apply. Tick `RUN_API` only when ready. For a provider requiring `max_tokens`, expand the code and change `CHAT_TOKEN_FIELD`.

In [ ]:
#@title 3. Connect the model API
API_PROTOCOL = "responses" #@param ["responses", "chat_completions"]
API_BASE_URL = "https://api.openai.com/v1" #@param {type:"string"}
MODEL = "" #@param {type:"string"}
RUN_API = False #@param {type:"boolean"}
# Advanced: change only if your provider specifies another token field or budget.
CHAT_TOKEN_FIELD = "max_completion_tokens"
MAX_OUTPUT_TOKENS = 4096
print("Enter the model ID and API settings supplied to you. Tick RUN_API when ready.")


## 4. Find promos
The model reviews your uploaded story. The code checks the returned timestamps, 20–60 second duration, complete episode coverage and uncut dialogue before producing the ranked report. If the model fails these checks, the run stops and keeps the response for inspection. It does not substitute a demo answer.

In [ ]:
#@title 4. Find and rank promos
import getpass
from IPython.display import HTML, display

report_path = None
run_folder = None
if not RUN_API:
    print("Enter the API settings and tick RUN_API in step 3, then run this cell.")
elif selected_input is None:
    print("Upload your screenplay in step 2 first.")
else:
    try:
        api_review.request_spec(API_PROTOCOL, API_BASE_URL, MODEL, "",
                                MAX_OUTPUT_TOKENS, CHAT_TOKEN_FIELD)
        api_key = getpass.getpass("API key (hidden): ")
        run_folder = Path("api_runs") / ("run_" + uuid.uuid4().hex[:12])
        try:
            report_path = api_review.review_with_api(selected_input, run_folder,
                api_key=api_key, model=MODEL, base_url=API_BASE_URL,
                protocol=API_PROTOCOL, max_episodes=episode_count,
                max_tokens=MAX_OUTPUT_TOKENS, chat_token_field=CHAT_TOKEN_FIELD)
        finally:
            del api_key
        display(HTML(report_path.read_text(encoding="utf-8")))
        print("Done. The report ranks promo moments from your uploaded story.")
        print("Outputs:", run_folder)
    except (ValueError, KeyError, TypeError, OSError) as error:
        print("Stopped:", str(error) if isinstance(error, api_review.APIReviewError) else
              "Input or model output failed validation. Check saved files; no valid report was produced.")


## 5. Download outputs
The ZIP contains the report, CSV, JSON, unedited model responses, source review packets and validation log. Keep outputs containing confidential scripts private.

In [ ]:
#@title 5. Download your results
import shutil

if report_path is None or not report_path.exists():
    print("Complete step 4 successfully before downloading results.")
else:
    archive = shutil.make_archive(str(run_folder), "zip", root_dir=run_folder)
    files.download(archive)


## Checks and limits
A strong hook, cliffhanger and standalone story are model judgments; verify them against the footage. Scores are editorial judgments, not predicted ad results. API and notebook control flow have been tested with simulated responses; a live provider call and hosted Colab upload have not been tested. Native provider APIs that do not follow either supported contract need another adapter.

The submitted 50-episode results are the saved original chat review. Running this notebook on a new input creates a separate fresh API review.

**No API key available?** Use the free chat-review route in the [README](https://github.com/atits23/cut-before-kuku-assignment#without-an-api-key). It uses the same prompts, JSON format and validator.

API references: [Responses/text generation](https://developers.openai.com/api/docs/guides/text) · [Chat Completions](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create).